# Court LLM Descriptor Extractor — Colab 96GB GPU Optimized 10k Run

This notebook performs only the GPU/LLM step:

`court_considerations.csv -> minimal semantic descriptors -> court_llm_descriptors_*.jsonl`

It intentionally does **not** produce final anchors, outcomes, retrieval views, or normalized production JSONL. Those are handled locally by `court_enrichment_normalizer.py`.

Optimized for a high-VRAM Colab/GCE GPU runtime. vLLM may reserve most GPU RAM for KV cache; that is expected when `gpu_memory_utilization` is high.


In [1]:
# Cell 0 — Colab setup and package install

import os
import sys
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

print('Running in Colab:', IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# vLLM is the primary runtime for AWQ inference here.
# After the first install in a fresh Colab runtime, restart the runtime once if imports fail.
!pip install -q -U "vllm>=0.8.5" "transformers>=4.51.0" accelerate safetensors pandas tqdm huggingface_hub

print('Setup done. If this was the first package install in a fresh Colab runtime, restart runtime once, then continue from Cell 1.')


Running in Colab: True
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup done. If this was the first package install in a fresh Colab runtime, restart runtime once, then continue from Cell 1.


In [2]:
# Cell 1 — Imports and runtime check

from __future__ import annotations

from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Any, Optional
from collections import Counter
import os
import sys
import re
import gc
import ast
import json
import time
import traceback

import pandas as pd
from tqdm.auto import tqdm

try:
    import torch
except Exception:
    torch = None

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

BASE_DIR = Path('/content/drive/MyDrive/swiss_law') if IN_COLAB else Path('..').resolve()
DATA_DIR = BASE_DIR / 'data'
MODEL_DIR = BASE_DIR / 'models'
OUTPUT_DIR = BASE_DIR / 'outputs'

print('Imports OK')
print('Running in Colab:', IN_COLAB)
print('BASE_DIR:', BASE_DIR)
print('DATA_DIR:', DATA_DIR)

if torch is not None and torch.cuda.is_available():
    print('CUDA devices:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        free, total = torch.cuda.mem_get_info(i)
        print(f'GPU {i}: {props.name}; free={free/1024**3:.2f} GiB / total={total/1024**3:.2f} GiB')
else:
    print('WARNING: CUDA GPU is not available. In Colab, select Runtime > Change runtime type > GPU.')


Imports OK
Running in Colab: True
BASE_DIR: /content/drive/MyDrive/swiss_law
DATA_DIR: /content/drive/MyDrive/swiss_law/data
CUDA devices: 1
GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition; free=94.43 GiB / total=94.97 GiB


In [3]:
# Cell 2 — Config

@dataclass
class Config:
    # Colab / Drive paths
    base_dir: str = str(BASE_DIR)
    data_dir: str = str(DATA_DIR)
    output_dir: str = str(OUTPUT_DIR)
    model_download_dir: str = str(MODEL_DIR / 'huggingface')

    # Input CSV. Put court_considerations.csv in /content/drive/MyDrive/swiss_law/data/
    input_csv: str = str(DATA_DIR / 'court_considerations.csv')
    fallback_input_csv: str = 'court_considerations.csv'

    # Hugging Face AWQ model id
    model_name: str = 'Qwen/Qwen3-8B-AWQ'

    # 10k pre-production shard
    start: int = 0
    limit: int = 10_000
    sample_random: bool = False
    random_seed: Optional[int] = 42
    min_text_chars: int = 80

    # Prompt budget. 2400 chars is usually enough for legal descriptors and keeps prefill fast.
    max_text_chars: int = 2400

    # GPU/vLLM.
    # For a single 96GB GPU, single-GPU mode is fastest and avoids TP communication.
    # If your runtime is actually 2x48GB and you want one sharded model, set gpu_mode='tp2',
    # but for this 8B AWQ model single-GPU is usually better.
    gpu_mode: str = 'single'   # 'single' or 'tp2'
    tensor_parallel_size: int = 1

    # vLLM preallocates KV cache up to this fraction. On 96GB, 0.82 may show ~75-85GB used.
    # Lower to 0.70 if you see OOM/instability; raise to 0.88 only if stable and batch needs it.
    gpu_memory_utilization: float = 0.82

    # Keep this tight; lower max_model_len = more KV capacity/concurrency.
    max_model_len: int = 3072

    # High-throughput settings for 96GB GPU. If OOM occurs, generation helper splits batches.
    max_num_seqs: int = 96
    batch_size: int = 64

    # False enables torch.compile/CUDA graphs in vLLM where supported; faster but may use extra memory.
    # If vLLM crashes during graph capture, set True and rerun.
    enforce_eager: bool = False

    # vLLM warned AWQ Marlin is faster for this model.
    quantization: str = 'awq_marlin'
    disable_custom_all_reduce: bool = True
    force_triton_attention: bool = False

    # Structured output is disabled because some vLLM versions can crash with guided/structured decoding.
    use_structured_outputs: bool = False

    # Minimal descriptor generation.
    # 320 is enough because we removed anchors, outcome, retrieval views, questions, and summaries.
    max_new_tokens: int = 320
    retry_max_new_tokens: int = 448
    max_retries: int = 1
    temperature: float = 0.0
    top_p: float = 1.0
    repetition_penalty: float = 1.02
    enable_thinking: bool = False

    # Output/debug
    include_raw_output_on_success: bool = False

cfg = Config()

# Apply GPU mode before loading vLLM.
# NOTE: If CUDA was already initialized before changing this, restart runtime.
if cfg.gpu_mode == 'single':
    os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
    cfg.tensor_parallel_size = 1
elif cfg.gpu_mode == 'tp2':
    os.environ.pop('CUDA_VISIBLE_DEVICES', None)
    cfg.tensor_parallel_size = 2
    cfg.disable_custom_all_reduce = True
else:
    raise ValueError("cfg.gpu_mode must be 'single' or 'tp2'")

if cfg.force_triton_attention:
    os.environ.setdefault('VLLM_ATTENTION_BACKEND', 'TRITON_ATTN')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

base_dir = Path(cfg.base_dir)
data_dir = Path(cfg.data_dir)
out_dir = Path(cfg.output_dir)
model_download_dir = Path(cfg.model_download_dir)
for d in [base_dir, data_dir, out_dir, model_download_dir]:
    d.mkdir(parents=True, exist_ok=True)

end_idx = cfg.start + cfg.limit - 1 if cfg.limit else -1
suffix = f'{cfg.start:07d}_{end_idx:07d}' if cfg.limit else f'{cfg.start:07d}_all'
output_jsonl = out_dir / f'court_llm_descriptors_{suffix}.jsonl'
output_preview_csv = out_dir / f'court_llm_descriptors_{suffix}_preview.csv'
output_failures_jsonl = out_dir / f'court_llm_descriptors_{suffix}_failures.jsonl'
output_metrics_json = out_dir / f'court_llm_descriptors_{suffix}_metrics.json'

print(json.dumps(asdict(cfg), indent=2))
print('Output JSONL:', output_jsonl)


{
  "base_dir": "/content/drive/MyDrive/swiss_law",
  "data_dir": "/content/drive/MyDrive/swiss_law/data",
  "output_dir": "/content/drive/MyDrive/swiss_law/outputs",
  "model_download_dir": "/content/drive/MyDrive/swiss_law/models/huggingface",
  "input_csv": "/content/drive/MyDrive/swiss_law/data/court_considerations.csv",
  "fallback_input_csv": "court_considerations.csv",
  "model_name": "Qwen/Qwen3-8B-AWQ",
  "start": 0,
  "limit": 10000,
  "sample_random": false,
  "random_seed": 42,
  "min_text_chars": 80,
  "max_text_chars": 2400,
  "gpu_mode": "single",
  "tensor_parallel_size": 1,
  "gpu_memory_utilization": 0.82,
  "max_model_len": 3072,
  "max_num_seqs": 96,
  "batch_size": 64,
  "enforce_eager": false,
  "quantization": "awq_marlin",
  "disable_custom_all_reduce": true,
  "force_triton_attention": false,
  "use_structured_outputs": false,
  "max_new_tokens": 320,
  "retry_max_new_tokens": 448,
  "max_retries": 1,
  "temperature": 0.0,
  "top_p": 1.0,
  "repetition_penalty"

In [4]:
# Cell 3 — Load CSV and select rows

def resolve_input_path() -> Path:
    candidates = [
        Path(cfg.input_csv),
        DATA_DIR / cfg.fallback_input_csv,
        BASE_DIR / cfg.fallback_input_csv,
        Path('/content') / cfg.fallback_input_csv,
        Path.cwd() / cfg.fallback_input_csv,
    ]
    for p in candidates:
        if p.exists():
            return p

    # Helpful fallback search for Colab/Drive and local runs.
    search_roots = [DATA_DIR, BASE_DIR, Path('/content')]
    for root in search_roots:
        if root.exists():
            for pat in ['**/court_considerations.csv', '**/court_consideration.csv']:
                found = sorted(root.glob(pat))
                if found:
                    return found[0]

    raise FileNotFoundError(
        'Could not find court_considerations.csv. Expected location: '
        f'{DATA_DIR / cfg.fallback_input_csv}'
    )

input_path = resolve_input_path()
print('Using input:', input_path)

df = pd.read_csv(input_path)
print('Shape:', df.shape)
print('Columns:', list(df.columns))

citation_col = 'citation' if 'citation' in df.columns else df.columns[0]
text_col = 'text' if 'text' in df.columns else df.columns[1]

valid = df[df[citation_col].notna() & df[text_col].notna()].copy()
valid[text_col] = valid[text_col].astype(str)
valid['_text_len'] = valid[text_col].str.strip().str.len()
valid = valid[valid['_text_len'] >= cfg.min_text_chars].copy()

if cfg.sample_random:
    pool = valid.iloc[cfg.start:] if cfg.start else valid
    work_df = pool.sample(n=min(cfg.limit, len(pool)), random_state=cfg.random_seed)
else:
    end = None if not cfg.limit else cfg.start + cfg.limit
    work_df = valid.iloc[cfg.start:end]

work_df = work_df.reset_index(drop=False).rename(columns={'index': '_source_row'})
print('Selected rows:', len(work_df))
display(work_df[['_source_row', citation_col, text_col, '_text_len']].head(20))


Using input: /content/drive/MyDrive/swiss_law/data/court_considerations.csv
Shape: (2476315, 2)
Columns: ['citation', 'text']
Selected rows: 10000


,_source_row,citation,text,_text_len
0,1,BGE 139 I 2 E. 2,Eventualiter sei die Rückweisung an die Vorins...,885
1,2,BGE 139 I 2 E. 5.1,"In der Sache ist vorweg zu prüfen, ob der Ents...",437
2,3,BGE 139 I 2 E. 5.2,Art. 34 Abs. 1 BV gewährleistet in allgemeiner...,242
3,4,BGE 139 I 2 E. 5.3,Im vorliegenden Fall geht es nicht um die Gült...,286
4,5,BGE 139 I 2 E. 7.1,S. 144) bestätigte das Verwaltungsgericht den ...,1493
5,6,BGE 139 I 2 E. 5.4,Strittig ist hier hingegen die Umsetzung der P...,183
6,7,BGE 139 I 2 E. 7.1,S. 70) dargestellten und im angefochtenen Ents...,690
7,8,BGE 139 I 2 E. 5.5,"Zu beachten ist sodann, dass nach der schwyzer...",904
8,9,BGE 139 I 2 E. 5.6,Die Umsetzung einer Planungsinitiative ist ver...,2405
9,10,BGE 139 I 2 E. 5.7,Die an der Volksabstimmung vom 26. November 20...,278


In [5]:
# Cell 4 — Minimal LLM descriptor schema and prompt

# The LLM intentionally does NOT produce anchors, outcomes, summaries, questions, or retrieval views.
# Local CPU scripts will build those deterministically later.
# Keep output compact: this is critical for 10k/2.4M-scale throughput.

DESCRIPTOR_KEYS = [
    'legal_area',
    'primary_domain',
    'secondary_domain',
    'legal_domain_path',
    'topic',
    'subtopic',
    'micro_topic',
    'concepts_en',
    'terms_original',
    'doctrinal_rule',
    'legal_test',
    'fact_pattern_tags',
    'procedural_context',
    'paragraph_role',
    'authority_role',
    'specificity_score',
]

ROLE_VALUES = {
    'holding', 'reasoning', 'facts', 'procedural_history', 'legal_standard',
    'application', 'citation', 'costs', 'notification', 'disposition', 'neutral'
}

# Short schema hint = fewer prompt tokens.
LLM_SCHEMA_HINT = {
    'legal_area': 'broad area, <=5 words',
    'primary_domain': '<=5 words',
    'secondary_domain': '<=7 words',
    'legal_domain_path': ['2-4 labels'],
    'topic': '<=7 words',
    'subtopic': '<=9 words',
    'micro_topic': '<=12 words; most specific issue',
    'concepts_en': ['3-5 English legal concepts'],
    'terms_original': ['3-6 exact source-language legal terms'],
    'doctrinal_rule': 'empty unless paragraph states rule; <=18 words',
    'legal_test': 'empty unless test/standard; <=16 words',
    'fact_pattern_tags': ['0-4 concrete tags'],
    'procedural_context': '<=8 words',
    'paragraph_role': 'holding|reasoning|facts|procedural_history|legal_standard|application|citation|costs|notification|disposition|neutral',
    'authority_role': ['0-2 labels'],
    'specificity_score': '0..1',
}

SYSTEM_PROMPT = '''You are a Swiss legal descriptor extractor.

Return exactly one compact JSON object.
Do not generate user questions or summaries.
Do not extract statute anchors, case anchors, outcomes, retrieval views, or quality flags.
Only extract semantic legal descriptors that static regex parsing cannot reliably infer.

Use English for classification fields.
Use exact German/French/Italian terms for terms_original.
If the paragraph is factual/procedural/boilerplate, keep doctrinal_rule and legal_test empty.
Prefer precise legal descriptors over generic words.
Keep all strings and arrays short.
JSON only.'''

USER_TEMPLATE = '''Citation: {citation}

Text:
{text}

Required JSON shape:
{schema}

Return JSON only.'''

def trim_text(text: str, max_chars: int) -> str:
    text = re.sub(r'\s+', ' ', str(text)).strip()
    if len(text) <= max_chars:
        return text
    head = max_chars // 2
    tail = max_chars - head
    return text[:head].rstrip() + ' ... [TRUNCATED] ... ' + text[-tail:].lstrip()

def build_user_prompt(citation: str, text: str) -> str:
    return USER_TEMPLATE.format(
        citation=str(citation),
        text=trim_text(text, cfg.max_text_chars),
        schema=json.dumps(LLM_SCHEMA_HINT, ensure_ascii=False, separators=(',', ':')),
    )


In [6]:
# Cell 5 — JSON parsing and descriptor normalization

def extract_json_object(raw: str) -> dict[str, Any]:
    if raw is None:
        raise ValueError('empty model output')
    s = str(raw).strip()
    s = re.sub(r'^\s*```(?:json)?\s*', '', s, flags=re.I)
    s = re.sub(r'\s*```\s*$', '', s)
    s = re.sub(r'<think>.*?</think>', '', s, flags=re.I | re.S).strip()

    start = s.find('{')
    if start < 0:
        raise ValueError(f'no JSON object start found: {s[:300]}')

    depth = 0
    in_str = False
    esc = False
    for i in range(start, len(s)):
        ch = s[i]
        if in_str:
            if esc:
                esc = False
            elif ch == '\\':
                esc = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == '{':
                depth += 1
            elif ch == '}':
                depth -= 1
                if depth == 0:
                    candidate = s[start:i+1]
                    try:
                        return json.loads(candidate)
                    except Exception:
                        candidate = re.sub(r',\s*([}\]])', r'\1', candidate)
                        try:
                            return json.loads(candidate)
                        except Exception:
                            return ast.literal_eval(candidate)
    raise ValueError(f'no balanced JSON object found: {s[:700]}')


def clean_str(x: Any, max_chars: int = 240) -> str:
    s = re.sub(r'\s+', ' ', str(x or '')).strip()
    return s[:max_chars].rstrip()


def clean_list(x: Any, max_items: int, max_chars: int = 80) -> list[str]:
    if x is None:
        return []
    if isinstance(x, str):
        x = [x]
    if not isinstance(x, (list, tuple, set)):
        return []
    out, seen = [], set()
    for item in x:
        s = clean_str(item, max_chars=max_chars)
        if not s:
            continue
        key = s.casefold()
        if key in seen:
            continue
        seen.add(key)
        out.append(s)
        if len(out) >= max_items:
            break
    return out


def normalize_descriptor(obj: dict[str, Any]) -> dict[str, Any]:
    d = {}
    d['legal_area'] = clean_str(obj.get('legal_area'), 80)
    d['primary_domain'] = clean_str(obj.get('primary_domain'), 80)
    d['secondary_domain'] = clean_str(obj.get('secondary_domain'), 100)
    d['legal_domain_path'] = clean_list(obj.get('legal_domain_path'), 4, 60)
    d['topic'] = clean_str(obj.get('topic'), 100)
    d['subtopic'] = clean_str(obj.get('subtopic'), 120)
    d['micro_topic'] = clean_str(obj.get('micro_topic'), 160)
    d['concepts_en'] = clean_list(obj.get('concepts_en'), 5, 70)
    d['terms_original'] = clean_list(obj.get('terms_original'), 6, 100)
    d['doctrinal_rule'] = clean_str(obj.get('doctrinal_rule'), 260)
    d['legal_test'] = clean_str(obj.get('legal_test'), 220)
    d['fact_pattern_tags'] = clean_list(obj.get('fact_pattern_tags'), 4, 70)
    d['procedural_context'] = clean_str(obj.get('procedural_context'), 120)

    role = clean_str(obj.get('paragraph_role'), 60).lower().replace(' ', '_').replace('-', '_')
    d['paragraph_role'] = role if role in ROLE_VALUES else 'neutral'
    d['authority_role'] = clean_list(obj.get('authority_role'), 2, 60)
    try:
        d['specificity_score'] = max(0.0, min(1.0, float(obj.get('specificity_score', 0))))
    except Exception:
        d['specificity_score'] = 0.0

    # Hard guarantee: forbidden/final fields never survive in raw descriptor.
    forbidden = {
        'statute_anchors', 'case_anchors', 'normalized_anchors', 'retrieval_views',
        'outcome_signal', 'query_phrases_en', 'natural_language_queries', 'legal_question',
        'summary_en', 'english_summary', 'enrichment_quality', 'anchor_quality_flags',
    }
    for k in forbidden:
        d.pop(k, None)
    return d


def empty_descriptor(error: str = '') -> dict[str, Any]:
    return {
        'legal_area': '',
        'primary_domain': '',
        'secondary_domain': '',
        'legal_domain_path': [],
        'topic': '',
        'subtopic': '',
        'micro_topic': '',
        'concepts_en': [],
        'terms_original': [],
        'doctrinal_rule': '',
        'legal_test': '',
        'fact_pattern_tags': [],
        'procedural_context': '',
        'paragraph_role': 'neutral',
        'authority_role': [],
        'specificity_score': 0.0,
        '_descriptor_error': error[:500],
    }


In [7]:
# Cell 6 — Load vLLM with Qwen3-8B-AWQ

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
import inspect

print('Loading tokenizer/model:', cfg.model_name)
if torch is not None and torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        print(f'Before vLLM load GPU {i}: free={free/1024**3:.2f} GiB total={total/1024**3:.2f} GiB')

tokenizer = AutoTokenizer.from_pretrained(
    cfg.model_name,
    trust_remote_code=True,
    cache_dir=cfg.model_download_dir,
)

base_llm_kwargs = dict(
    model=cfg.model_name,
    trust_remote_code=True,
    tensor_parallel_size=cfg.tensor_parallel_size,
    gpu_memory_utilization=cfg.gpu_memory_utilization,
    max_model_len=cfg.max_model_len,
    max_num_seqs=cfg.max_num_seqs,
    enforce_eager=cfg.enforce_eager,
    disable_custom_all_reduce=cfg.disable_custom_all_reduce,
    disable_log_stats=True,
    download_dir=cfg.model_download_dir,
)

# Add optional speed flags only if this vLLM build supports them.
try:
    sig = inspect.signature(LLM)
    if 'enable_prefix_caching' in sig.parameters:
        base_llm_kwargs['enable_prefix_caching'] = True
except Exception:
    pass

# vLLM quantization names can differ across versions/builds. Try requested mode first.
quantization_candidates = []
for q in [cfg.quantization, 'awq_marlin', 'awq', None]:
    if q not in quantization_candidates:
        quantization_candidates.append(q)

last_error = None
for quantization in quantization_candidates:
    llm_kwargs = dict(base_llm_kwargs)
    if quantization is not None:
        llm_kwargs['quantization'] = quantization
    try:
        print(f'Initializing vLLM with quantization={quantization!r}')
        llm = LLM(**llm_kwargs)
        print('vLLM loaded')
        if torch is not None and torch.cuda.is_available():
            for i in range(torch.cuda.device_count()):
                free, total = torch.cuda.mem_get_info(i)
                print(f'After vLLM load GPU {i}: free={free/1024**3:.2f} GiB total={total/1024**3:.2f} GiB')
        break
    except Exception as exc:
        last_error = exc
        print(f'vLLM load failed with quantization={quantization!r}: {repr(exc)}')
        gc.collect()
        if torch is not None and torch.cuda.is_available():
            torch.cuda.empty_cache()
else:
    raise RuntimeError('Could not load Qwen3-8B-AWQ with vLLM') from last_error


Loading tokenizer/model: Qwen/Qwen3-8B-AWQ
Before vLLM load GPU 0: free=94.43 GiB total=94.97 GiB
Initializing vLLM with quantization='awq_marlin'
INFO 05-03 08:38:58 [utils.py:233] non-default args: {'trust_remote_code': True, 'download_dir': '/content/drive/MyDrive/swiss_law/models/huggingface', 'max_model_len': 3072, 'gpu_memory_utilization': 0.82, 'max_num_seqs': 96, 'disable_log_stats': True, 'quantization': 'awq_marlin', 'disable_custom_all_reduce': True, 'model': 'Qwen/Qwen3-8B-AWQ'}


config.json: 0.00B [00:00, ?B/s]

INFO 05-03 08:39:08 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 05-03 08:39:08 [nixl_utils.py:34] NIXL is not available
WARNING 05-03 08:39:08 [nixl_utils.py:44] NIXL agent config is not available
INFO 05-03 08:39:08 [model.py:555] Resolved architecture: Qwen3ForCausalLM
INFO 05-03 08:39:08 [model.py:1680] Using max model len 3072
INFO 05-03 08:39:08 [awq_marlin.py:252] The model is convertible to awq_marlin during runtime. Using awq_marlin kernel.
INFO 05-03 08:39:08 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=16384.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Parse safetensors files:   0%|          | 0/2 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

WARNING 05-03 08:39:12 [system_utils.py:157] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
vLLM loaded
After vLLM load GPU 0: free=17.14 GiB total=94.97 GiB


In [8]:
# Cell 7 — Generation helpers

def render_prompt(citation: str, text: str, repair: bool = False, bad_output: str = '', error: str = '') -> str:
    user_prompt = build_user_prompt(citation, text)
    if repair:
        user_prompt = f'''The previous output was invalid JSON.

Parser error:
{error}

Previous output:
{bad_output[:1400]}

Repair by returning exactly one complete compact JSON object using the same schema.
Do not add questions, summaries, anchors, outcomes, or retrieval views.

{user_prompt}'''

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_prompt},
    ]
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=cfg.enable_thinking,
        )
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate_raw(prompts: list[str], max_tokens: int) -> list[str]:
    params = SamplingParams(
        temperature=cfg.temperature,
        top_p=cfg.top_p,
        max_tokens=max_tokens,
        repetition_penalty=cfg.repetition_penalty,
    )
    outputs = llm.generate(prompts, sampling_params=params, use_tqdm=False)
    return [out.outputs[0].text if out.outputs else '' for out in outputs]


def generate_raw_safe(prompts: list[str], max_tokens: int, min_split: int = 1) -> list[str]:
    """Generate a batch; on OOM/runtime failure, recursively split instead of falling back to all single rows."""
    try:
        return generate_raw(prompts, max_tokens)
    except Exception as exc:
        if len(prompts) <= min_split:
            raise
        mid = len(prompts) // 2
        print(f'Batch of {len(prompts)} failed ({type(exc).__name__}); splitting into {mid}+{len(prompts)-mid}')
        left = generate_raw_safe(prompts[:mid], max_tokens, min_split=min_split)
        right = generate_raw_safe(prompts[mid:], max_tokens, min_split=min_split)
        return left + right


def generate_descriptor(citation: str, text: str, first_raw: str | None = None) -> tuple[dict[str, Any], dict[str, Any]]:
    attempts = []
    raw = first_raw
    for attempt in range(cfg.max_retries + 1):
        try:
            if raw is None:
                raw = generate_raw_safe([render_prompt(citation, text)], cfg.max_new_tokens)[0]
            obj = extract_json_object(raw)
            desc = normalize_descriptor(obj)
            return desc, {
                'status': 'ok' if attempt == 0 else 'ok_after_retry',
                'attempt_count': attempt + 1,
                'error': None,
                'raw_output': raw if cfg.include_raw_output_on_success else None,
            }
        except Exception as exc:
            err = repr(exc)
            attempts.append({'attempt': attempt + 1, 'error': err, 'raw_output': (raw or '')[:1400]})
            if attempt >= cfg.max_retries:
                return empty_descriptor(err), {
                    'status': 'failed_descriptor_parse',
                    'attempt_count': attempt + 1,
                    'error': err,
                    'attempts': attempts,
                    'raw_output': raw,
                }
            raw = generate_raw_safe([render_prompt(citation, text, repair=True, bad_output=raw or '', error=err)], cfg.retry_max_new_tokens)[0]


In [9]:
# Cell 8 — Run descriptor extraction and write raw LLM descriptor JSONL

records_for_preview = []
failures = []
status_counter = Counter()
t0 = time.time()
processed = 0

# Stream JSONL as we go, so a long 10k run does not lose all progress if interrupted.
with output_jsonl.open('w', encoding='utf-8') as out_f:
    for start in tqdm(range(0, len(work_df), cfg.batch_size), desc='llm-descriptor batches'):
        batch = work_df.iloc[start:start + cfg.batch_size]
        row_objs = []
        prompts = []
        for _, row in batch.iterrows():
            citation = str(row[citation_col])
            text = str(row[text_col])
            row_obj = {
                '_source_row': int(row['_source_row']),
                'citation': citation,
                'text': text,
            }
            row_objs.append(row_obj)
            prompts.append(render_prompt(citation, text))

        try:
            raws = generate_raw_safe(prompts, cfg.max_new_tokens)
        except Exception as exc:
            print('Batch generation failed completely; falling back to per-row retries:', repr(exc))
            raws = [None] * len(row_objs)

        for row_obj, raw in zip(row_objs, raws):
            desc, gen = generate_descriptor(row_obj['citation'], row_obj['text'], first_raw=raw)
            rec = {
                '_source_row': row_obj['_source_row'],
                'citation': row_obj['citation'],
                'text': row_obj['text'],
                'llm_enrichment': desc,
                'llm_generation': {
                    'model': cfg.model_name,
                    'method': 'minimal_descriptor_only',
                    **gen,
                },
            }
            out_f.write(json.dumps(rec, ensure_ascii=False) + '\n')
            processed += 1
            status_counter[gen['status']] += 1

            # Keep preview in memory, not all 10k rows.
            if len(records_for_preview) < 1000:
                records_for_preview.append(rec)
            if gen['status'].startswith('failed'):
                failures.append(rec)

        # Flush every batch.
        out_f.flush()

elapsed = time.time() - t0

if failures:
    with output_failures_jsonl.open('w', encoding='utf-8') as f:
        for rec in failures:
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
else:
    if output_failures_jsonl.exists():
        output_failures_jsonl.unlink()

preview_rows = []
for rec in records_for_preview:
    e = rec['llm_enrichment']
    g = rec['llm_generation']
    preview_rows.append({
        '_source_row': rec['_source_row'],
        'citation': rec['citation'],
        'status': g['status'],
        'legal_area': e.get('legal_area'),
        'primary_domain': e.get('primary_domain'),
        'secondary_domain': e.get('secondary_domain'),
        'topic': e.get('topic'),
        'subtopic': e.get('subtopic'),
        'micro_topic': e.get('micro_topic'),
        'concepts_en': ' | '.join(e.get('concepts_en', [])),
        'terms_original': ' | '.join(e.get('terms_original', [])),
        'doctrinal_rule': e.get('doctrinal_rule'),
        'legal_test': e.get('legal_test'),
        'fact_pattern_tags': ' | '.join(e.get('fact_pattern_tags', [])),
        'procedural_context': e.get('procedural_context'),
        'paragraph_role': e.get('paragraph_role'),
        'authority_role': ' | '.join(e.get('authority_role', [])),
        'specificity_score': e.get('specificity_score'),
    })
preview_df = pd.DataFrame(preview_rows)
preview_df.to_csv(output_preview_csv, index=False)

metrics = {
    'start': cfg.start,
    'limit': cfg.limit,
    'selected_rows': len(work_df),
    'written_rows': processed,
    'failures': len(failures),
    'elapsed_seconds': elapsed,
    'rows_per_second': processed / max(elapsed, 1e-9),
    'status_counts': dict(status_counter),
    'output_jsonl': str(output_jsonl),
    'output_preview_csv': str(output_preview_csv),
    'output_failures_jsonl': str(output_failures_jsonl) if failures else None,
    'config': asdict(cfg),
}
output_metrics_json.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')

print(json.dumps(metrics, indent=2))
display(preview_df.head(50))


llm-descriptor batches:   0%|          | 0/157 [00:00<?, ?it/s]

{
  "start": 0,
  "limit": 10000,
  "selected_rows": 10000,
  "written_rows": 10000,
  "failures": 0,
  "elapsed_seconds": 759.1911044120789,
  "rows_per_second": 13.171914083139905,
  "status_counts": {
    "ok": 9998,
    "ok_after_retry": 2
  },
  "output_jsonl": "/content/drive/MyDrive/swiss_law/outputs/court_llm_descriptors_0000000_0009999.jsonl",
  "output_preview_csv": "/content/drive/MyDrive/swiss_law/outputs/court_llm_descriptors_0000000_0009999_preview.csv",
  "output_failures_jsonl": null,
  "config": {
    "base_dir": "/content/drive/MyDrive/swiss_law",
    "data_dir": "/content/drive/MyDrive/swiss_law/data",
    "output_dir": "/content/drive/MyDrive/swiss_law/outputs",
    "model_download_dir": "/content/drive/MyDrive/swiss_law/models/huggingface",
    "input_csv": "/content/drive/MyDrive/swiss_law/data/court_considerations.csv",
    "fallback_input_csv": "court_considerations.csv",
    "model_name": "Qwen/Qwen3-8B-AWQ",
    "start": 0,
    "limit": 10000,
    "sample_rand

,_source_row,citation,status,legal_area,primary_domain,secondary_domain,topic,subtopic,micro_topic,concepts_en,terms_original,doctrinal_rule,legal_test,fact_pattern_tags,procedural_context,paragraph_role,authority_role,specificity_score
0,1,BGE 139 I 2 E. 2,ok,Administrative law,Administrative review,Remand for reconsideration,Remand decision,Remand for re-examination,Remand for re-examination and reconsideration,remand | reconsideration | administrative revi...,Rückweisung | Neubehandlung | Sachverhaltsabkl...,,,administrative appeal | public decision | judi...,Administrative appeal,reasoning,Federal Court | Administrative Court,0.8
1,2,BGE 139 I 2 E. 5.1,ok,Constitutional Law,Federal Constitution,Popular Initiative,Constitutional Compatibility,Popular Initiative and Local Decision,Compatibility of municipal decision with popul...,constitutional compatibility | popular initiat...,Volksentscheid | Gemeinderatsentscheid | Art. ...,,Compliance with Art. 34 BV,popular initiative | municipal decision | cons...,Constitutional review,reasoning,Federal Court | Constitutional Law,0.8
2,3,BGE 139 I 2 E. 5.2,ok,Constitutional Law,Federal Constitution,Local Autonomy,Political Rights,Initiative Rights,Initiative Rights in Municipal Matters,Political Rights | Initiative Rights | Local A...,Art. 34 Abs. 1 BV | Initiativrecht | Kommunale...,,,Constitutional Provision | Local Governance,Constitutional Interpretation,reasoning,Constitutional Court,0.8
3,4,BGE 139 I 2 E. 5.3,ok,Constitutional Law,Initiatives,Validity of Initiatives,Initiative Validity,Initiative validity and proceedings,Initiative validity and prior proceedings,Initiative validity | Administrative law | Jud...,Initiative | Abstimmung | Verwaltungsgericht,,,Prior proceedings | Administrative court | Ini...,Prior administrative court proceedings,procedural_history,Administrative court,0.7
4,5,BGE 139 I 2 E. 7.1,ok,Administrative law,Local governance,Initiative procedures,Initiative validity,Constitutional compatibility,Initiative compliance with higher law,initiative validity | constitutional review | ...,GOG | GOG/SRSZ | PBG/SZ | SRSZ,,,initiative review | higher law | community dec...,Community initiative review,reasoning,Swiss cantonal authority | judicial review,0.8
5,6,BGE 139 I 2 E. 5.4,ok,Administrative law,Planning law,Implementation of plans,Plan implementation,Dispute over plan execution,Dispute over implementation of planning initia...,plan implementation | administrative dispute |...,Planungsinitiative | Verwaltungsgericht | VGE III,,,plan dispute | administrative court | previous...,Administrative court decision,reasoning,administrative court,0.3
6,7,BGE 139 I 2 E. 7.1,ok,Administrative law,Planning law,Public participation,Participation rights,Right to object during planning,Right to object after administrative review,public participation | right to object | admin...,Nutzungsplanerlassverfahren | Einsprachebefugn...,,,planning initiative | right to object | admini...,Administrative review process,reasoning,Swiss cantonal court,0.8
7,8,BGE 139 I 2 E. 5.5,ok,Local Government,Municipal Law,Voting Procedures,Voting Rights,Amendment of Planning Decisions,Amending zoning changes at municipal assembly,municipal assembly | zoning plans | amendment ...,Gemeindeversammlung | Urnenabstimmung | Zonen-...,,,municipal decision | amendment | voting proces...,Municipal assembly voting,reasoning,Swiss Federal Court | Swiss cantonal court,0.8
8,9,BGE 139 I 2 E. 5.6,ok,Constitutional Law,Initiatives,Implementation and Interpretation,Initiative Implementation,Constitutional Compliance and Interpretation,Comparative analysis of initiative implementat...,constitutional compliance | initiative interpr...,Gesetzesinitiative | Verfassungsinitiative | P...,,,unformulated initiative | constitutional compl...,Constitutional interpretation,reasoning,Constitutional Court | Schweizerisches Bundesg...,0.8
9,10,BGE 139 I 2 E. 5.7,ok,Constitutional law,Local governance,Land use planning,Zoning chang

In [10]:
# Cell 9 — QC: verify this is raw descriptor output only

FORBIDDEN = {
    'statute_anchors', 'case_anchors', 'normalized_anchors', 'retrieval_views',
    'outcome_signal', 'query_phrases_en', 'natural_language_queries', 'legal_question',
    'summary_en', 'english_summary', 'enrichment_quality', 'anchor_quality_flags',
}

def find_forbidden(obj: Any, path: str = '') -> list[str]:
    hits = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            p = f'{path}.{k}' if path else k
            if k in FORBIDDEN:
                hits.append(p)
            hits.extend(find_forbidden(v, p))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            hits.extend(find_forbidden(v, f'{path}[{i}]'))
    return hits

qc = []
with output_jsonl.open(encoding='utf-8') as f:
    for i, line in enumerate(f):
        if not line.strip():
            continue
        rec = json.loads(line)
        e = rec['llm_enrichment']
        qc.append({
            'citation': rec['citation'],
            'status': rec['llm_generation']['status'],
            'forbidden_fields': find_forbidden(rec),
            'concept_count': len(e.get('concepts_en', [])),
            'terms_original_count': len(e.get('terms_original', [])),
            'has_topic': bool(e.get('topic') or e.get('subtopic') or e.get('micro_topic')),
            'specificity_score': e.get('specificity_score'),
        })
qc_df = pd.DataFrame(qc)
display(qc_df.head(100))
print('Rows checked:', len(qc_df))
print('Forbidden field rows:', int(qc_df['forbidden_fields'].apply(bool).sum()))
print('Failed rows:', int(qc_df['status'].str.startswith('failed').sum()))
print('Status counts:', qc_df['status'].value_counts().to_dict())


,citation,status,forbidden_fields,concept_count,terms_original_count,has_topic,specificity_score
0,BGE 139 I 2 E. 2,ok,[],5,4,True,0.8
1,BGE 139 I 2 E. 5.1,ok,[],3,3,True,0.8
2,BGE 139 I 2 E. 5.2,ok,[],3,3,True,0.8
3,BGE 139 I 2 E. 5.3,ok,[],3,3,True,0.7
4,BGE 139 I 2 E. 7.1,ok,[],5,4,True,0.8
...,...,...,...,...,...,...,...
95,BGE 136 I 1 E. 5.4.3,ok,[],5,5,True,0.8
96,BGE 136 I 1 E. 5.4.4,ok,[],5,4,True,0.8
97,BGE 136 I 1 E. 5.5.1,ok,[],3,3,True,0.8
98,BGE 136 I 1 E. 5.5.2,ok,[],4,4,True,0.8


Rows checked: 10000
Forbidden field rows: 0
Failed rows: 0
Status counts: {'ok': 9998, 'ok_after_retry': 2}


## Next local step

The notebook writes outputs under:

```text
/content/drive/MyDrive/swiss_law/outputs/
```

Use the generated `court_llm_descriptors_*.jsonl` as input to the local finalizer, which should call:

```python
normalize_enriched_court_row(
    citation=citation,
    text=text,
    llm_enrichment=raw["llm_enrichment"],
    deterministic_metadata=metadata,
)
```

That local step creates the final production JSONL with:

```text
rag_enrichment
normalized_anchors
anchor_quality_flags
retrieval_views
enrichment_quality
```
